In [28]:
from src.server.models.dms_model import DmsModel
from src.server.models.platform_values.models.platform_values_helper import PlatformValuesCalcHelper
from src.db import DbConnection
import numpy as np
import pandas as pd

conn = DbConnection()

page = 0
limit = 100

arr = []

tags_rows = []
rows = conn.select_all("""
select p.tag_id_int, m.tag_id
  from app.def_sheet_platform_product p, app.map_tag_id m
  where p.tag_id_int = m.id
  and p.is_prompt = 1
  order by m.tag_id
""")
for row in rows:
    tags_rows.append(row["tag_id"])

while True:
    print(f"Page: {page}")

    PlatformValuesCalcHelper.recreate_table(
        conn=conn,
        table_name="tmp_part",
        query=f"""
        CREATE TABLE platform_values.x__table_name__x (
        app_id_int int unsigned not null,
        app_id_in_store tinytext not null,
        is_steam int unsigned not null,
        constraint x__table_name__x_pk
        primary key (app_id_int)
        )
        SELECT e.app_id_int, m.app_id_in_store, IF(e.store = 1, 0, 1) as is_steam
          FROM scraped_data.apps e,
          app.map_app_id_to_store_id m
          where e.app_id_int = m.app_id_int
          and e.released is not null
          and e.installs > 1000
         order by e.app_id_int
         limit {page * limit}, {limit}
        """
    )
    cnt = conn.select_one(f"""
    select count(1) as cnt from platform_values.tmp_part
    """)["cnt"]

    if cnt == 0:
        break

    page += 1

    rows = conn.select_all("""
    select part.app_id_int, m.tag_id, part.app_id_in_store, part.is_steam
      from platform_values.tmp_part part,
      platform.platform_values_apps a,
      platform.platform_values_tags t,
      app.map_tag_id m
     where part.app_id_int = a.app_id_int
     and a.app_id_int = t.app_id_int
     and m.id = t.tag_id_int
    """)

    tags_per_app_id_int = {}
    app_id_per_app_id_int = {}
    for row in rows:
        app_id_int = row["app_id_int"]
        if app_id_int not in app_id_per_app_id_int:
            app_id_per_app_id_int[app_id_int] = {
                "app_id_in_store": row["app_id_in_store"],
                "is_steam": row["is_steam"]
            }
        if app_id_int not in tags_per_app_id_int:
            tags_per_app_id_int[app_id_int] = [0] * len(tags_rows)
        tag_id = row["tag_id"]
        ix = tags_rows.index(tag_id)
        tags_per_app_id_int[app_id_int][ix] = 1

    for k in tags_per_app_id_int:
        arr.append([k, app_id_per_app_id_int[k]["app_id_in_store"], app_id_per_app_id_int[k]["is_steam"]] + tags_per_app_id_int[k])

    #break

df1 = pd.DataFrame(np.array(arr), columns = ['app_id_int', 'app_id_in_store', 'is_steam'] + tags_rows)
guid = "1f80e518-431b-4c8d-b5d0-942bceeaabd5"
DmsModel.write_df_to_external_data(conn=conn, guid=guid, df=df1)
conn.commit()

page = 0
limit = 100

arr = []

while True:
    print(f"Page: {page}")

    PlatformValuesCalcHelper.recreate_table(
        conn=conn,
        table_name="tmp_part",
        query=f"""
        CREATE TABLE platform_values.x__table_name__x (
        app_id_int int unsigned not null,
        app_id_in_store tinytext not null,
        constraint x__table_name__x_pk
        primary key (app_id_int)
        )
        SELECT e.app_id_int, m.app_id_in_store
          FROM scraped_data.ext_data e,
          app.map_app_id_to_store_id m
          where e.app_id_int = m.app_id_int
         order by e.app_id_int
         limit {page * limit}, {limit}
        """
    )
    cnt = conn.select_one(f"""
    select count(1) as cnt from platform_values.tmp_part
    """)["cnt"]

    if cnt == 0:
        break

    page += 1

    ret_arpu_per_app = {}
    app_id_per_app_id_int = {}
    rows_arpu_ret = conn.select_all("""
    select e.app_id_int, e.app_id_in_store, e.d28_ret_gp, e.revenues_gp_ww, e.downloads_gp_ww
      FROM platform_values.tmp_part part,
           scraped_data.ext_data e
     WHERE part.app_id_int = e.app_id_int
    """)
    for row in rows_arpu_ret:
        arr.append([row["app_id_int"], row["app_id_in_store"]] + [
            row["d28_ret_gp"], row["revenues_gp_ww"], row["downloads_gp_ww"]
        ])

    #break

df2 = pd.DataFrame(np.array(arr), columns = ['app_id_int', "app_id_in_store", "d28_ret_gp", "revenues_gp_ww", "downloads_gp_ww"])
guid = "930e43d4-e352-4574-a07c-efdad0da93f6"
DmsModel.write_df_to_external_data(conn=conn, guid=guid, df=df2)
conn.commit()
conn.close()
df1

Page: 0
Page: 1
Page: 2
Page: 3
Page: 4
Page: 5
Page: 6
Page: 7
Page: 8
Page: 9
Page: 10
Page: 11
Page: 12
Page: 13
Page: 14
Page: 15
Page: 16
Page: 17
Page: 18
Page: 19
Page: 20
Page: 21
Page: 22
Page: 23
Page: 24
Page: 25
Page: 26
Page: 27
Page: 28
Page: 29
Page: 30
Page: 31
Page: 32
Page: 33
Page: 34
Page: 35
Page: 36
Page: 37
Page: 38
Page: 39
Page: 40
Page: 41
Page: 42
Page: 43
Page: 44
Page: 45
Page: 46
Page: 47
Page: 48
Page: 49
Page: 50
Page: 51
Page: 52
Page: 53
Page: 54
Page: 55
Page: 56
Page: 57
Page: 58
Page: 59
Page: 60
Page: 61
Page: 62
Page: 63
Page: 64
Page: 65
Page: 66
Page: 67
Page: 68
Page: 69
Page: 70
Page: 71
Page: 72
Page: 73
Page: 74
Page: 75
Page: 76
Page: 77
Page: 78
Page: 79
Page: 80
Page: 81
Page: 82
Page: 83
Page: 84
Page: 85
Page: 86
Page: 87
Page: 88
Page: 89
Page: 90
Page: 91
Page: 92
Page: 93
Page: 94
Page: 95
Page: 96
Page: 97
Page: 98
Page: 99
Page: 100
Page: 101
Page: 102
Page: 103
Page: 104
Page: 105
Page: 106
Page: 107
Page: 108
Page: 109
Page: 110


,app_id_int,app_id_in_store,is_steam,activity__high,activity__low,activity__medium,automation__high,automation__low,automation__medium,behaviors__bond,...,visuals__2d,visuals__3d,visuals__first_person,visuals__high,visuals__isometric,visuals__low,visuals__medium,visuals__side,visuals__third_person,visuals__top_down
0,1,com.kabam.marvelbattle,0,0,0,1,0,1,0,1,...,0,1,0,1,0,0,0,0,0,0
1,2,com.kabam.bullseye,0,0,1,0,0,1,0,1,...,0,1,0,1,0,0,0,0,0,0
2,3,com.ripostegames.shopr,0,0,1,0,0,0,1,1,...,0,1,0,1,0,0,0,0,0,0
3,4,com.tripledot.woodoku,0,0,1,0,0,1,0,0,...,1,0,0,0,0,1,0,0,0,0
4,5,com.tripledot.triple.tile.match.pair.game.thre...,0,0,1,0,0,1,0,0,...,0,1,0,1,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16093,29573,steam__356450,1,0,1,0,1,0,0,0,...,1,0,0,0,0,0,1,0,0,0
16094,29575,steam__968550,1,0,0,1,0,1,0,0,...,0,1,0,1,1,0,0,0,0,0
16095,29577,steam__897930,1,0,1,0,0,1,0,1,...,1,0,0,1,0,0,0,0,0,0
16096,29578,steam__1719660,1,0,1,0,0,1,0,0,...,1,1,0,1,0,0,0,0,0,0


In [26]:
conn.close()

In [30]:
df2

,app_id_int,app_id_in_store,d28_ret_gp,revenues_gp_ww,downloads_gp_ww
0,1,com.kabam.marvelbattle,13.51,597453602,119565648
1,2,com.kabam.bullseye,8.06,7854211,4712447
2,3,com.ripostegames.shopr,26.83,36693943,2702340
3,4,com.tripledot.woodoku,11.68,25761,67175523
4,7,com.tripledot.blackjack,12.90,442592,3300710
...,...,...,...,...,...
7288,15397,air.bullstudios.fairyhorsecare,-100.00,173944,25497043
7289,15406,com.boyagames.fishing,-100.00,308544,308023
7290,15407,com.volvapps.ef3.x.admob,-100.00,998620,586724
7291,15408,com.neopop.guitargirl.match3.puzzle,-100.00,123151,114707
